<a href="https://colab.research.google.com/github/kuds/reinforce-tactics/blob/claude/cool-thompson-rYUIt/notebooks/kaggle_environments_smoke_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reinforce Tactics — Kaggle Environments Smoke Test

End-to-end validation that the **Reinforce Tactics** environment works inside `kaggle-environments`, pulled **straight from the [`kuds/kaggle-environments`](https://github.com/kuds/kaggle-environments) fork** (the repo hosting the upstream PR) rather than copied in from this project.

This notebook:
1. Clones a branch of `kuds/kaggle-environments` and installs `kaggle-environments` from it, so the `reinforce_tactics` env code comes directly from that repo.
2. Runs a series of `make()` / `reset()` / `run()` smoke tests through the public API.
3. Exercises the vendored engine directly for a few sanity checks.
4. Runs the functional test suite that ships in the repo (`tests/envs/reinforce_tactics/`).

The repo + branch are exposed as parameters in the first code cell, so you can point the notebook at **any branch** to test it before it merges upstream.

## 1. Setup

Set the repo + branch you want to test below, clone it, and install `kaggle-environments` from the clone.

We install with `--no-deps` and add only `numpy` + `pandas` (all the `reinforce_tactics` env needs). The full package declares heavy dependencies for *other* environments (JAX, open-spiel, pokerkit, …); skipping them keeps setup fast. Importing `kaggle_environments` then prints a one-line `Loading environment … failed` notice for each of those unrelated envs — expected and harmless here. To install everything instead, replace the install line with `pip install -q <REPO_DIR>`.

In [ ]:
# === Parameters =============================================================
# Point the smoke test at any branch of the kaggle-environments fork.
# Change KE_BRANCH to test a different branch before it merges upstream.
KE_REPO = "https://github.com/kuds/kaggle-environments.git"
KE_BRANCH = "claude/review-reinforce-tactics-l68Ob"
# ===========================================================================

print(f"Testing {KE_REPO} @ {KE_BRANCH}")

In [ ]:
import subprocess

REPO_DIR = "/tmp/kaggle-environments"

# Clone the requested branch.
subprocess.run(["rm", "-rf", REPO_DIR], check=False)
subprocess.run(
    ["git", "clone", "--depth", "1", "-b", KE_BRANCH, KE_REPO, REPO_DIR],
    check=True,
)

# Install kaggle-environments straight from the clone. --no-deps avoids the heavy
# dependencies that *other* envs need; reinforce_tactics only needs numpy + pandas,
# which we add explicitly alongside pytest for the suite.
subprocess.run(["pip", "install", "-q", "--no-deps", REPO_DIR], check=True)
subprocess.run(["pip", "install", "-q", "numpy", "pandas", "pytest"], check=True)
print("Installed kaggle-environments from", REPO_DIR)

Importing `kaggle_environments` scans the `envs/` directory and registers every environment, including `reinforce_tactics`. The unrelated envs that need optional dependencies are skipped automatically.

In [ ]:
import contextlib
import io

# Capture the import-time "Loading environment ... failed" chatter from unrelated
# envs that need optional deps, then summarise it in one line.
_buf = io.StringIO()
with contextlib.redirect_stdout(_buf):
    from kaggle_environments import evaluate, make

skipped = [
    line.split()[2]
    for line in _buf.getvalue().splitlines()
    if line.startswith("Loading environment")
]
print("reinforce_tactics registered via kaggle_environments.make().")
if skipped:
    print(f"Skipped {len(skipped)} unrelated env(s) (optional deps): {', '.join(skipped)}")

## 2. Smoke tests via the public API

`make("reinforce_tactics")` returns a working `Environment`. Reset it, inspect the starting state, then run a full episode.

In [ ]:
env = make(
    "reinforce_tactics",
    configuration={"mapName": "beginner", "episodeSteps": 30, "mapSeed": 42},
    debug=False,
)
state = env.reset()

obs = state[0]["observation"]
print("Number of agent slots :", len(state))
print("Player 0 status       :", state[0]["status"])
print("Player 1 status       :", state[1]["status"])
print("Starting gold (P0/P1) :", obs["gold"])
print("Map name (config)     :", env.configuration.mapName)
print("Board size            :", len(obs["board"]), "x", len(obs["board"][0]))

structures = obs["structures"]
print("Structure count       :", len(structures))
print("Structure types       :", sorted({s["type"] for s in structures}))

In [ ]:
# ASCII render of the initial board
print(env.render(mode="ansi"))

In [ ]:
# Run a full episode: the built-in random agent vs the aggressive agent.
result = env.run(["random", "aggressive"])
final = result[-1]

print(f"Turns simulated      : {len(result) - 1}")
print(f"Final status (P0/P1) : {final[0].status} / {final[1].status}")
print(f"Final reward (P0/P1) : {final[0].reward} / {final[1].reward}")

if final[0].reward == final[1].reward:
    print("Outcome              : Draw or step-cap reached")
elif final[0].reward > final[1].reward:
    print("Outcome              : random agent won")
else:
    print("Outcome              : aggressive agent won")

## 3. Configuration variations

Spot-check the configurable knobs: the built-in maps, random generation, fog of war, custom starting gold, and the `evaluate()` helper.

In [ ]:
# Every built-in map should reset cleanly to a >=20x20 board (small maps are padded).
from kaggle_environments.envs.reinforce_tactics.reinforce_tactics import BUILTIN_MAPS

for name in sorted(BUILTIN_MAPS):
    e = make("reinforce_tactics", configuration={"mapName": name})
    s = e.reset()
    rows = len(s[0]["observation"]["board"])
    cols = len(s[0]["observation"]["board"][0])
    assert rows >= 20 and cols >= 20, (name, rows, cols)
    print(f"  {name:<18} -> {rows}x{cols} board, status={s[0]['status']}")
print(f"  all {len(BUILTIN_MAPS)} built-in maps reset OK")

In [ ]:
# Random map (mapName='') and fog of war.
rand_env = make("reinforce_tactics", configuration={"mapName": "", "mapSeed": 7})
rand_state = rand_env.reset()
ocean_count = sum(row.count("o") for row in rand_state[0]["observation"]["board"])
print(f"Random map ocean tiles    : {ocean_count}")

fog_env = make("reinforce_tactics", configuration={"fogOfWar": True, "mapSeed": 42})
fog_state = fog_env.reset()
print(f"Fog of war units list type: {type(fog_state[0]['observation']['units']).__name__}")

rich_env = make("reinforce_tactics", configuration={"startingGold": 1000, "mapSeed": 42})
rich_state = rich_env.reset()
print(f"Custom starting gold      : {rich_state[0]['observation']['gold']}")

In [ ]:
# evaluate(): play several episodes and collect rewards.
rewards = evaluate(
    "reinforce_tactics",
    ["random", "random"],
    num_episodes=3,
    configuration={"mapSeed": 42, "episodeSteps": 20},
)
print("Per-episode rewards (each row sums to 0):")
for i, r in enumerate(rewards):
    print(f"  Episode {i + 1}: P0={r[0]}, P1={r[1]}")

## 4. Engine-direct sanity checks

The kaggle interpreter doesn't exercise every engine code path. The cells below import the vendored engine directly from the installed package and confirm a few invariants:

* `get_legal_actions()` returns a dict and exposes `create_unit` at a building.
* Legal moves exclude tiles occupied by the moving player's own units.
* `resign()` ends the game and assigns the win to the opponent.
* `to_dict()` / `from_dict()` round-trips the units on the board.

In [ ]:
import numpy as np
import pandas as pd

from kaggle_environments.envs.reinforce_tactics.reinforce_tactics_engine import GameState


def small_map():
    m = np.array([["p"] * 10 for _ in range(10)], dtype=object)
    m[0, 0] = "h_1"
    m[0, 1] = "b_1"
    m[9, 9] = "h_2"
    m[9, 8] = "b_2"
    return pd.DataFrame(m)


# Legal actions: returns a dict and includes create_unit at the building.
g = GameState(small_map(), num_players=2)
actions = g.get_legal_actions()
assert isinstance(actions, dict)
assert any((a["x"], a["y"]) == (1, 0) for a in actions["create_unit"])
print("legal_actions OK -> create_unit at building exposed")

# Move filter: an ally-occupied tile must NOT appear as a legal destination.
g2 = GameState(small_map(), num_players=2)
g2.player_gold = {1: 5000, 2: 5000}
ally_a = g2.create_unit("W", 4, 4, player=1)
g2.create_unit("W", 5, 4, player=1)
ally_a.can_move = True
moves = [m for m in g2.get_legal_actions(player=1)["move"] if m["unit"] is ally_a]
destinations = {(m["to_x"], m["to_y"]) for m in moves}
assert (5, 4) not in destinations
print("move_filter OK -> ally-occupied tile excluded")

In [ ]:
# resign() ends the game and the opponent wins.
g3 = GameState(small_map(), num_players=2)
g3.player_gold = {1: 5000, 2: 5000}
g3.create_unit("W", 3, 3, player=1)
g3.create_unit("M", 6, 6, player=2)
g3.resign(player=1)
assert g3.game_over and g3.winner == 2
print(f"resign OK -> game_over={g3.game_over}, winner={g3.winner}")

# to_dict / from_dict round-trips the units on the board.
g4 = GameState(small_map(), num_players=2)
g4.player_gold = {1: 5000, 2: 5000}
g4.create_unit("W", 4, 3, player=1)
g4.create_unit("M", 6, 6, player=2)
before = sorted((u.type, u.x, u.y, u.player) for u in g4.units)
restored = GameState.from_dict(g4.to_dict(), small_map())
after = sorted((u.type, u.x, u.y, u.player) for u in restored.units)
assert before == after, (before, after)
print(f"save_load_round_trip OK -> {len(after)} units preserved: {after}")

## 5. Run the functional test suite

The repo ships a functional suite at **`tests/envs/reinforce_tactics/test_reinforce_tactics.py`** — 21 tests that exercise the env through `kaggle_environments.make()`. We run it straight from the clone.

In [ ]:
result = subprocess.run(
    [
        "python",
        "-m",
        "pytest",
        "tests/envs/reinforce_tactics/test_reinforce_tactics.py",
        "-v",
        "-p",
        "no:cacheprovider",
    ],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
# Drop the unrelated env-load warnings from the captured output.
out = "\n".join(
    line
    for line in result.stdout.splitlines()
    if not line.startswith("Loading environment")
)
print(out)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise SystemExit(f"functional suite failed with exit code {result.returncode}")
print("\nFunctional suite: PASS")

## Done

If every cell ran without errors, `kaggle_environments.make("reinforce_tactics")` — installed directly from the `kuds/kaggle-environments` branch you selected — works through the public API, the engine-direct sanity checks pass, and the 21-test functional suite is green.

**Next steps:**
* Change `KE_BRANCH` at the top to validate a different branch before it merges upstream.
* Browse `kaggle_environments/envs/reinforce_tactics/README.md` in the clone for the action / observation reference.
* Submit your own agent: write `def agent(observation, configuration) -> list[dict]` and pass it to `env.run([my_agent, "random"])`.